___
# Week 4, notebook 1: Data Menipulation

> **Directions:**
> 
> * Step 1. Click **"Run all"** at the top to execute the whole notebook.
>
> * Step 2. Go down the page following directions below. 
>
> * Do not re-run the cells. If unsure, click on the little downward pointing arrow next to "Run all" and select "Restart session".
>
> Enjoy!

## Key learning:

After working through this notebook you will be familiar with:

* The structure of the dataset
* Different types of variables contained in the dataset
* Descriptive statistics
* Number of missing values
* Number of outliers
* Types of encoding for categorical variables

## Import libraries

Here, we import open-source libraries that we will use for data transformation.
* `pandas` is a powerful and flexible framework for working with datasets
* `ipywidgets` enables interactive visualisations
* `utils` is out custom script that provides some additional functions

In [1]:
# Import libraries
import pandas as pd
from ipywidgets import (
    Dropdown,
    Button,
    Layout,
    Output,
    GridspecLayout,
    Label, 
    FloatSlider, 
    HBox,
    interact,
)
import hvplot.pandas

from utils import get_cols_for_display, apply_filter_mask, make_venn_diagram, find_outliers

# Set display options and button color
pd.set_option('display.max_columns', None)
pd.set_option('display.max_colwidth', None)
button_color = 'lightblue'

## Load data

Data for type 2 diabetes patients has been pre-selected and saved in a separate file. 
* `parquet` files allow storing large amounts of data and associated metadata and quickly loading them in Jupyter notebooks. 
* The dataset contains patient demographics​, vitals​, edical history, including history and family history of diabetes​, problem list​, current medications​, lab results​, number of previous admissions.
* All data is synthetic and does not contain any real patient data.

In [2]:
# Load the prepared dataset with only T2DM patients
dataset = pd.read_parquet("../data/t2dm_synthetic_T2DM_only_prepared.parquet", engine="pyarrow")

### Skim the dataset
Have a quick look at the first 10 rows of the dataset:
* What columns does it contain?
* What type of values can we see in the columns?

In [3]:
# Show the first 15 rows of the dataset
dataset.head(15)

,patient_id,hospital,age,sex,height,weight,weight_over_100,bmi,waist,obesity,interpreter,alcohol,smoker,systolic_bp,diastolic_bp,elevated_bp,years_since_diagnosis,gestational_diabetes,family_history,siblings,children,hypertension,hyperlipidaemia,ischemic_heart_disease,cardiac_failure,neuropathy,nephropathy,sympt_peripheral_neuropathy,depression,stroke,acute_myocardial_infarction,transient_ischemic_attack,cabg,cardiomyopathy,autonomic_neuropathy,retinopathy,lower_limb_problems,charcot_foot,ulceration,claudication,nephropathy_indication,cardiovascular_disease,cerebrovascular_disease,oral_contraceptive,beta_blockers,ace_inhibitor,calcium_channel_blocker,corticosteroids,thaizide,agr2_receptor_blocker,aspirin,method_manage_t2dm,sodium,potassium,chloride,bicarbonate,creatinine,urea,haemoglobin,albumin,white_cell_count,platelets,red_cell_count,packed_cell_volume,mean_cell_volume,hba1c,egfr,islet_antibody,coeliac_antibody,thyroid_function,thyroid_antibody,vitamin_b12,dka_diagnosis,admissions_in_2024,visits_before_2025,outcome_admission_in_2025
0,922904,WH,18,F,173.0,107.9,True,36.047673,91.9,True,False,False,NO,NaN,NaN,False,5,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,NON-INSULIN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,9.2,NaN,NaN,NaN,NaN,NaN,NaN,True,False,True,False,False,False,0,2,False
1,118725,RMH,19,M,158.0,89.9,False,36.006682,84.5,True,False,False,UNKNOWN,110.0,80.1,False,7,False,True,False,False,False,False,False,False,True,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,TABLETS,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,8.3,NaN,NaN,NaN,NaN,NaN,NaN,False,False,False,False,False,False,0,3,False
2,888078,RMH,19,F,169.0,134.9,True,47.227716,105.7,True,False,False,UNKNOWN,119.0,70.1,False,2,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,DIET ONLY,136.0,4.4,103.9,22.7,83.2,4.1,149.4,41.0,7.7,408.0,5.5,0.4,85.1,10.0,NaN,False,False,False,False,False,False,12,14,False
3,811787,WH,20,M,167.0,NaN,False,NaN,NaN,False,False,False,CURRENT,NaN,NaN,False,2,False,True,False,False,False,False,True,False,False,False,False,False,False,True,False,False,False,False,False,False,False,False,False,False,True,False,False,False,False,False,False,False,False,False,INSULIN,139.0,4.1,109.9,30.9,56.2,3.8,117.8,39.0,NaN,230.0,5.9,0.4,85.1,6.9,NaN,True,True,True,True,False,False,0,2,False
4,334671,WH,28,F,162.0,NaN,False,NaN,NaN,False,True,True,CURRENT,NaN,NaN,False,11,False,True,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,True,False,False,False,False,False,False,False,False,False,False,False,False,TABLETS,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,True,False,False,False,False,False,0,12,True
5,407486,WH,24,F,145.0,85.9,False,40.849979,84.9,True,False,False,NO,NaN,NaN,False,1,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,NON-INSULIN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,10.6,NaN,NaN,NaN,NaN,NaN,NaN,False,False,False,False,False,False,0,1,False
6,876505,WH,29,F,161.0,NaN,False,NaN,NaN,False,False,False,FORMER,NaN,NaN,False,10,False,False,False,False,False,False,True,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,INSULIN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,True,False,False,False,False,False,0,10,True
7,188919,RMH,30,F,159.0,119.2,True,47.157881,91.8,True,False,False,CURRENT,NaN,NaN,False,1,Fal

## Examine column types

Here, we can look at variable types and understand what information is stored in each column.
* Variables are grouped by type: discrete, continuous, boolean, or categorical. Use the drop down menu to pick a variable type.
* Browse through column names and their definitions.
* You may have to rely on your clinician team members to fully understand what each column means.

In [4]:
# Map column names to types and descriptions for display
cols_for_display = get_cols_for_display(dataset)

# Interactive widget to select a type of feature and display the corresponding columns
data_types_list = Dropdown(
    options=['Discrete', 'Continuous', 'Boolean', 'Categorical', 'All'],
    description='Select a variable type:',
    disabled=False,
    style={'description_width': 'max-content'},
    value='Discrete'
)

def retrieve_cols_for_display(col_type):
    if col_type != 'All':
        return cols_for_display[(cols_for_display['type'] == col_type)]
    else:
        return cols_for_display

interact(retrieve_cols_for_display, col_type=data_types_list);

interactive(children=(Dropdown(description='Select a variable type:', options=('Discrete', 'Continuous', 'Bool…

## Subset records

Let's now practice boolean logic and slice our dataset by two variables.
* Pick two columns that you would like to include in your criteria
* Define the criteria, e.g. would you like to focus on female patients?
* Define the boolean logic to combine the two criteria

In [5]:
# Operator symbol mapping: display label → Python operator string
operator_dict = {"=": "==", "≥": ">=", ">": ">", "!=": "!=", "<": "<", "≤": "<="}
# Reverse mapping: Python operator string → display label (used for Venn labels)
r_operator_dict = dict(zip(operator_dict.values(), operator_dict.keys()))

# --- Widget definitions ---
# Left operand widgets
widget_dataset = Dropdown(description='Table:', options=['dataset'])
widget_col1 = Dropdown(description='Column 1:', options=dataset.columns, value='sex')
widget_col1_op = Dropdown(description='Operator:', options=operator_dict)
widget_col1_value = Dropdown(description='Value:', options=sorted(dataset[widget_col1.value].unique()))
# Right operand widgets
widget_col2 = Dropdown(description='Column 2:', options=dataset.columns, value='method_manage_t2dm')
widget_col2_op = Dropdown(description='Operator:', options=operator_dict)
widget_col2_value = Dropdown(description='Value:', options=sorted(dataset[widget_col2.value].unique()))
# Boolean logic widget
boolean_logic = Dropdown(description='Boolean condition:', options=['AND (∩)', 'OR (∪)'], style={'description_width': 'max-content'})
# Action button
button1 = Button(description='Show result', layout=Layout(width='100%'))
button1.style.button_color = button_color

# --- Event handlers: refresh value dropdown when column selection changes ---
def update_value_options(val_dropdown, change):
    val_dropdown.options = sorted(dataset[change.new].unique())

widget_col1.observe(lambda change: update_value_options(widget_col1_value, change), names=['value'])
widget_col2.observe(lambda change: update_value_options(widget_col2_value, change), names=['value'])

output1 = Output()

# --- Button handler: apply filters, plot Venn diagram, display results ---
def on_button_clicked1(_):
    pd.set_option('display.max_rows', 50)
    output1.clear_output()

    if widget_col1_value.value is None or widget_col2_value.value is None:
        return

    # Build boolean masks using operator functions (avoids eval)
    mask1 = apply_filter_mask(dataset, widget_col1.value, widget_col1_op.value, widget_col1_value.value)
    mask2 = apply_filter_mask(dataset, widget_col2.value, widget_col2_op.value, widget_col2_value.value)

    is_and = boolean_logic.value == 'AND (∩)'
    combined_mask = mask1 & mask2 if is_and else mask1 | mask2
    result_dataset = dataset[combined_mask]

    # Patient keys matching each individual condition and their intersection
    subs1 = dataset.loc[mask1, 'patient_id'].tolist()
    subs2 = dataset.loc[mask2, 'patient_id'].tolist()
    inter = list(set(subs1) & set(subs2))

    # Labels for Venn diagram sets
    label1 = f"{widget_col1.value} {r_operator_dict[widget_col1_op.value]} {widget_col1_value.value}"
    label2 = f"{widget_col2.value} {r_operator_dict[widget_col2_op.value]} {widget_col2_value.value}"

    with output1:
        if is_and:
            make_venn_diagram(subs1, subs2, inter, label1, label2, colors=('purple', 'skyblue'))
            print('Number of rows in result:', len(inter))
        else:
            make_venn_diagram(subs1, subs2, inter, label1, label2, colors=('skyblue', 'skyblue'))
            print('Number of rows in result:', result_dataset.shape[0])
        display(result_dataset)

button1.on_click(on_button_clicked1)

# --- Grid layout (3 rows × 3 cols) ---
grid1 = GridspecLayout(3, 3)
grid1[0, 0] = widget_col1
grid1[1, 0] = widget_col1_op
grid1[2, 0] = widget_col1_value
grid1[0, 1] = boolean_logic
grid1[0, 2] = widget_col2
grid1[1, 2] = widget_col2_op
grid1[2, 2] = widget_col2_value
grid1[2, 1] = button1

display(grid1)
display(output1)

GridspecLayout(children=(Dropdown(description='Column 1:', index=3, layout=Layout(grid_area='widget001'), opti…

Output()

## Descriptive statistics

Descriptive statistics is one of the first steps in exploratory data analysis and can reveal a lot of information.
* For numeric variables (discrete and continuous), it is useful to look at the mean and standard deviation, minimum and maximum values, as well as quartiles.
* For categorical and boolean variables, the number of unique values, the most commonly occuring value and its frequency are the key statistics to focus on.

In [6]:
# Interactive widget to generate descriptive statistics for discrete and continuous variables
button2 = Button(description ='Generate descriptive statistics for discrete and continuous variables', layout=Layout(width='50%'))
button2.style.button_color = button_color
output2 = Output()

def on_button_clicked2(_):
    output2.clear_output()
    with output2:
        display(dataset.drop('patient_id', axis=1).describe(include=['int64', 'float64']).T.head(50))

button2.on_click(on_button_clicked2)

display(button2)
display(output2)

Button(description='Generate descriptive statistics for discrete and continuous variables', layout=Layout(widt…

Output()

In [7]:
# Interactive widget to generate descriptive statistics for categorical and boolean variables
button3 = Button(description ='Generate descriptive statistics for categorical and boolean variables', layout=Layout(width='50%'))
button3.style.button_color = button_color
output3 = Output()

def on_button_clicked3(_):
    output3.clear_output()
    with output3:
        display(dataset.describe(include=['category', 'bool']).T.head(50))

button3.on_click(on_button_clicked3)

display(button3)
display(output3)

Button(description='Generate descriptive statistics for categorical and boolean variables', layout=Layout(widt…

Output()

## Missing values

Now let's have a look at how much data in our dataset is missing:
* There may be different reasons for why data is missing.
* Sometimes a value can be calcualted from other variables in the dataset.
* Other times a variable may have to be excluded from further analysis altogether.

In [8]:
# Calculate the number of missing values in each column and sort in descending order
missing_values = dataset.isna().sum().sort_values(ascending=False)
missing_values = missing_values[missing_values > 0]
missing_values = missing_values.to_frame(name='Number of missing values')

# Interactive widget to show the number of missing values in each column
button4 = Button(description ='Show number of missing values', layout=Layout(width='50%'))
button4.style.button_color = button_color
output4 = Output()

def on_button_clicked4(_):
    output4.clear_output()
    n_values = dataset.shape[0]
    missing_values = dataset.isna().sum().sort_values(ascending=False)
    missing_values = missing_values[missing_values > 0]
    missing_values = missing_values.to_frame(name='Number of missing values')
    missing_values['Proportion of missing values'] = (missing_values['Number of missing values'] / n_values * 100).round(2)
    with output4:
        display(missing_values)

button4.on_click(on_button_clicked4)

display(button4)
display(output4)

Button(description='Show number of missing values', layout=Layout(width='50%'), style=ButtonStyle(button_color…

Output()

### Remove variables with missing values
One approach is to remove variables for which the proportion of missing values exceeds a given threshold.
* Try adjusting the threshold and examine the remaining columns in the dataset.
* Discuss with your team why using a blanket rule like this may not be optimal.

In [9]:
label = Label('Proportion of missing values:')
w = FloatSlider(value=0.5, min=0, max=1, step=0.05)
display(HBox([label, w]))

button5 = Button(description ='Remove variables with missing values',layout=Layout(width='50%'))
button5.style.button_color = button_color
output5 = Output()

def on_button_clicked5(_):
    output5.clear_output()
    n_values = dataset.shape[0]
    with output5:
        thresh = n_values * (1 - w.value)
        print("Removing columns with the proportion of missing values above %.0f%%" % (w.value * 100))
        print('Number of columns in the original dataset: ' + str(dataset.shape[1]))
        print('Number of columns after deletion: ' + str(dataset.dropna(axis=1, thresh=thresh).shape[1]))

        missing_values = dataset.dropna(axis=1, thresh=thresh).isna().sum().sort_values(ascending=False)
        # missing_values = missing_values[missing_values > 0]
        missing_values = missing_values.to_frame(name='Number of missing values')
        missing_values['Proportion of missing values'] = (missing_values['Number of missing values'] / n_values * 100).round(2)
        display(missing_values)

button5.on_click(on_button_clicked5)

display(button5)
display(output5)

Button(description='Remove variables with missing values', layout=Layout(width='50%'), style=ButtonStyle(butto…

Output()

## Examine outliers
Outliers are observations that dare substantially different from the rest of the values.
* Here, we assume a standard distribution and calculate outliers using the inter-quartile range (IQR) formula.
* Just like with missing values, there may be different reasons for why a value is an outlier.
* Often, it is a data entry error. This is why it is important to build rules into information systems to flag such errors when data is being entered.
* With your team, examine barplots and hsitograms for discrete and continuous variables in the dataset. This may be a handy tool to confirm your earlier suspicions.

In [10]:
# Interactive widget to select a column and display a boxplot and histogram with outliers highlighted
num_vars_list = Dropdown(
    options=dataset.drop('patient_id', axis=1).select_dtypes(include=['int64', 'float64']).columns, 
    description='Select a column to view:', 
    disabled=False,
    style={'description_width': 'max-content'}, 
    value = 'years_since_diagnosis'
    )

def retrieve_col_for_display(col):
    outliers = find_outliers(dataset[col])
    n_outliers = len(outliers)
    print('Column "' + col + '" plotted with outliers')
    print('Number of outliers:', n_outliers)
    return dataset.hvplot.box(col,  width=450) + dataset.hvplot.hist(col,  width=450)

interact(retrieve_col_for_display, col=num_vars_list);

interactive(children=(Dropdown(description='Select a column to view:', index=7, options=('age', 'height', 'wei…

### Filter outliers
Now let's plot the same variables after filtering the outliers.
* For which variables there is a notable change in how the values are distributed?
* Note, some of the barplots might still show outliers. This is because the plotting function re-calculates outliers based on the new distribution. 

In [ ]:
# Interactive widget to select a column and display a boxplot and histogram without outliers
button6 = Button(description ='Filter outliers', layout=Layout(width='50%'))
button6.style.button_color = button_color
output6 = Output()

def on_button_clicked6(_):
    output6.clear_output()
    outliers = find_outliers(dataset[num_vars_list.value])
    n_outliers = len(outliers)
    dataset_new = dataset[~dataset[num_vars_list.value].isin(outliers)]
    with output6:
        print('Column "' + num_vars_list.value + '" plotted without outliers')
        print('Number of outliers removed:', n_outliers)
        display(dataset_new.hvplot.box(num_vars_list.value,  width=450) + dataset_new.hvplot.hist(num_vars_list.value,  width=450))

button6.on_click(on_button_clicked6)
display(button6)
display(output6)

Button(description='Filter outliers', layout=Layout(width='50%'), style=ButtonStyle(button_color='lightblue'))

Output()

## Convert categorical variables to numbers

Recall that we have a few categorical variables in our dataset. 
* Categorical variables are incredibly useful when dealing with a finite number of possible values. 
* Common examples of categorical variables are seasons (spring, summer, autumn, winter), sex/gender variables, diagnosis (either as a string or as an ICD code).
* In `pandas`, categorical variables are treated in a special way: We can define a set of possible values, and if we later try to add a new category, it will be automatically replaced by NULL.

However, machine learning algorithms, whether it is a simple logistic regression or a large language model, only work with numeric inputs. So when using categorical features, like the ones above, we need to convert categories to integers. 

In [ ]:
# Interactive widget to select a categorical variable and display the value counts
cat_vars_list = Dropdown(
    options = dataset.select_dtypes(include='category'),
    description ='Categorical features:',
    style={'description_width': 'max-content'}, 
    value = 'hospital')

def retrieve_value_counts(col):
    return dataset[col].value_counts().to_frame(name='Number of values')

interact(retrieve_value_counts, col=cat_vars_list);

interactive(children=(Dropdown(description='Categorical features:', options=('hospital', 'sex', 'smoker', 'met…

### Simple encoding
The simplest approach is to take the set of N possible categories and map it to a set of integers, from 0 to N-1. 
* Our dataset contains only a few categorical varibles, check the results of encoding for each variable. 
* When does it NOT make sense to use simple encoding? 

In [14]:
# Interactive widget to show simple encoding of a categorical variable
button6 = Button(description ='Simple encoding of a categorical variable',layout=Layout(width='50%'))
button6.style.button_color = button_color
output6 = Output()

def on_button_clicked6(_):
    output6.clear_output()
    mapping  = dict(enumerate(dataset[cat_vars_list.value].cat.categories))
    with output6:
        print("Mapping categories to numbers:")
        for code, category in mapping.items():
            print(f"{category} is mapped to {code}")
        print()
        display(pd.concat([
            dataset.patient_id,
            dataset[cat_vars_list.value],
            pd.Series(dataset[cat_vars_list.value].cat.codes, name=cat_vars_list.value + '_encoded')
            ], axis=1).head(10))

button6.on_click(on_button_clicked6)
display(button6)
display(output6)

Button(description='Simple encoding of a categorical variable', layout=Layout(width='50%'), style=ButtonStyle(…

Output()

### One-hot encoding

Another approach is to create a boolean variable for each category.
* In the example with seasons, this would mean creating four boolean variables, for example, `is_spring`, `is_summer`, `is_autumn`, `is_winter`.
* In fact, only three boolean variables would suffice: if `is_spring`, `is_summer`, and `is_autumn` are all set to `False`, we know with certainty that it is winter. 
* The benefit of one-hot encoding is that we are not emposing any order on the categories.
* Have a look at how one-hot encoding applies to our categorical variables. What could be the downsides of it? 

In [15]:
# Interactive widget to show one-hot encoding of a categorical variable
button7 = Button(description ='One-hot encoding of a categorical variable',layout=Layout(width='50%'))
button7.style.button_color = button_color
output7 = Output()

def on_button_clicked7(_):
    output7.clear_output()
    with output7:
        display(pd.concat([
            dataset.patient_id,
            dataset[cat_vars_list.value],
            pd.get_dummies(dataset[cat_vars_list.value])
            ], axis=1).head(10))

button7.on_click(on_button_clicked7)
display(button7)
display(output7)

Button(description='One-hot encoding of a categorical variable', layout=Layout(width='50%'), style=ButtonStyle…

Output()